# Google Cloud Console Setup: Get Your OAuth Credentials

Before we can add "Login with Google" to our app, we need to register our app with Google. This notebook walks through every step.

**What we need by the end:**
- `GOOGLE_CLIENT_ID` — your app's public identifier at Google
- `GOOGLE_CLIENT_SECRET` — your app's private password at Google
- A redirect URI configured: `http://localhost:8000/auth/callback`

**Time:** ~10 minutes

# Step 1: Go to Google Cloud Console

Open this URL in your browser:

**https://console.cloud.google.com**

Sign in with your Google account if prompted.

You'll land on the Google Cloud dashboard.

# Step 2: Create a New Project

1. Click the **project dropdown** at the top of the page (next to "Google Cloud")
2. Click **"New Project"** in the top right of the popup
3. Fill in:
   - **Project name:** `AgentFlow` (or any name you like)
   - **Organization:** leave as default
4. Click **"Create"**
5. Wait a few seconds, then **select the new project** from the dropdown

```
Why do we need a project?
Google organizes everything by project.
Your OAuth credentials, APIs, billing — all belong to a project.
Think of it like a folder for your app's Google stuff.
```

# Step 3: Configure the OAuth Consent Screen

Before creating credentials, Google needs to know what your app looks like to users.

1. In the left sidebar, go to **APIs & Services → OAuth consent screen**
2. Click **"Get Started"** or **"Configure Consent Screen"**
3. Choose **User Type:**
   - Select **"External"** (allows any Google account to log in)
   - Click **"Create"**
4. Fill in the required fields:
   - **App name:** `AgentFlow`
   - **User support email:** select your email
   - **Developer contact email:** your email
5. Click **"Save and Continue"**

**Scopes page:**
1. Click **"Add or Remove Scopes"**
2. Check these three:
   - `openid`
   - `userinfo.email`
   - `userinfo.profile`
3. Click **"Update"** then **"Save and Continue"**

**Test users page:**
1. Click **"Add Users"**
2. Add your own Gmail address (and any others you want to test with)
3. Click **"Save and Continue"**

Click **"Back to Dashboard"**

```
Why "External" and "Testing"?
While the app is in "Testing" mode, only the emails you add
as test users can log in. This is fine for development.
To let anyone log in, you'd publish the app (requires Google review).
```

# Step 4: Create OAuth 2.0 Credentials

This is the step where you get the `CLIENT_ID` and `CLIENT_SECRET`.

1. In the left sidebar, go to **APIs & Services → Credentials**
2. Click **"+ Create Credentials"** at the top
3. Select **"OAuth client ID"**
4. Fill in:
   - **Application type:** `Web application`
   - **Name:** `AgentFlow Web` (or anything)
5. Under **"Authorized redirect URIs":**
   - Click **"+ Add URI"**
   - Enter exactly: `http://localhost:8000/auth/callback`
   - This MUST match what's in your code. If it doesn't, Google will reject the login.
6. Click **"Create"**

A popup appears with your credentials:
- **Client ID:** something like `123456789-abcdef.apps.googleusercontent.com`
- **Client Secret:** something like `GOCSPX-abcdefgh123456`

**Copy both values. You'll need them in the next step.**

```
What is the redirect URI?
After the user logs in at Google, Google needs to know
WHERE to send the user back. That's this URL.
Google will redirect to: http://localhost:8000/auth/callback?code=abc123
Your FastAPI server receives the code at this endpoint.
```

# Step 5: Add Credentials to Your .env File

Open the `.env` file in the agent-flow project and paste your credentials:

In [ ]:
# Run this cell to check if your .env has Google credentials set

from dotenv import load_dotenv
import os

load_dotenv("../../.env")  # adjust path if running from a different location
load_dotenv()              # also try current directory

client_id = os.getenv("GOOGLE_CLIENT_ID", "")
client_secret = os.getenv("GOOGLE_CLIENT_SECRET", "")

if not client_id:
    print("GOOGLE_CLIENT_ID is EMPTY")
    print("Open .env and add: GOOGLE_CLIENT_ID=your-client-id-here")
elif not client_id.endswith(".apps.googleusercontent.com"):
    print(f"GOOGLE_CLIENT_ID looks wrong: {client_id[:30]}...")
    print("It should end with .apps.googleusercontent.com")
else:
    print(f"GOOGLE_CLIENT_ID: {client_id[:20]}...{client_id[-30:]}")
    print("Looks correct!")

print()

if not client_secret:
    print("GOOGLE_CLIENT_SECRET is EMPTY")
    print("Open .env and add: GOOGLE_CLIENT_SECRET=your-secret-here")
elif len(client_secret) < 10:
    print(f"GOOGLE_CLIENT_SECRET looks too short: {client_secret}")
else:
    print(f"GOOGLE_CLIENT_SECRET: {client_secret[:8]}...{client_secret[-4:]}")
    print("Looks correct!")

if client_id and client_secret:
    print("\nBoth credentials are set. You're ready to test Google login!")

# Step 6: Test It

Start the backend and frontend:

```bash
# Terminal 1
cd agent-flow
python -m uvicorn app:app --reload --port 8000

# Terminal 2
cd agent-flow/frontend
npm run dev
```

Then:
1. Open `http://localhost:5173`
2. Click "Start chatting"
3. Click "Continue with Google"
4. Google login page appears
5. Sign in with your Google account
6. You're redirected back to AgentFlow, logged in with your real name and picture

If you see an error like "redirect_uri_mismatch":
- Go back to Google Console → Credentials → Edit your OAuth client
- Make sure the redirect URI is **exactly** `http://localhost:8000/auth/callback`
- No trailing slash, no https, no different port

# Quick Reference: What You Created

| Thing | Where it lives | What it is |
|-------|---------------|------------|
| **Project** | Google Cloud Console | Container for all your app's Google stuff |
| **OAuth Consent Screen** | APIs & Services → OAuth consent screen | What users see when they log in ("AgentFlow wants to access your email") |
| **OAuth Client ID** | APIs & Services → Credentials | Your app's public identifier (safe to expose) |
| **OAuth Client Secret** | APIs & Services → Credentials | Your app's private password (NEVER expose in frontend) |
| **Redirect URI** | Inside the OAuth Client config | Where Google sends users after login: `http://localhost:8000/auth/callback` |
| **Test users** | OAuth consent screen → Test users | Who can log in while app is in testing mode |

```
Where each credential ends up in your code:

.env file:
  GOOGLE_CLIENT_ID=123456789-abc.apps.googleusercontent.com
  GOOGLE_CLIENT_SECRET=GOCSPX-abc123

config.py:
  GOOGLE_CLIENT_ID = os.getenv("GOOGLE_CLIENT_ID")      reads from .env
  GOOGLE_CLIENT_SECRET = os.getenv("GOOGLE_CLIENT_SECRET")  reads from .env

auth/routes.py:
  oauth = OAuth2Session(client_id=GOOGLE_CLIENT_ID, ...)   uses it to talk to Google
```